# Autoencoder features

**Iteration 5** — representation learning, added after the iteration-4 round.

A denoising autoencoder on the **internal** feature space → a compressed embedding plus reconstruction error (a nonlinear anomaly signal). Unsupervised (never sees the target or `EXT_SOURCE`), prefixed `x_ae_`; saved to `autoencoder.pkl`. Trains on MPS if available.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from src.data import load_master, save_features

X, _ = load_master(all_rows=True)
idx = X.index
drop = [c for c in X.columns if c.lower().startswith(("ext_", "x_kmeans_", "x_pca_", "x_ae_")) or "ext_source" in c.lower()]
internal = X.drop(columns=drop)
del X                                               # free the full master immediately
keep = internal.columns[(internal.std(numeric_only=True) > 0) & (internal.isna().mean() < 0.9)]
internal = internal[keep]

def prep(df):
    df = df.astype("float32")
    med = df.median().astype("float32")             # keep everything float32 - no float64 upcast
    mean = df.mean().astype("float32")
    std = df.std().astype("float32").replace(0, 1)
    df = df.fillna(med)
    return ((df - mean) / std).fillna(0).to_numpy("float32")

Z = prep(internal)
del internal                                        # free the frame; keep only the float32 array
dev = "mps" if torch.backends.mps.is_available() else "cpu"
Z.shape, dev

((356255, 3224), 'mps')

## Train

In [2]:
class AE(nn.Module):
    def __init__(self, d, bottleneck=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, bottleneck))
        self.dec = nn.Sequential(nn.Linear(bottleneck, 128), nn.ReLU(), nn.Linear(128, d))
    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), z

torch.manual_seed(0)
model = AE(Z.shape[1]).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
model.train()
n = len(Z)
for epoch in range(20):
    tot = 0.0
    perm = np.random.RandomState(epoch).permutation(n)
    for i in range(0, n, 4096):
        xb = torch.tensor(Z[perm[i:i + 4096]], device=dev)
        rec, _ = model(xb + 0.1 * torch.randn_like(xb))
        loss = loss_fn(rec, xb)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(xb)
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  loss {tot / n:.4f}")

epoch  0  loss 0.4978
epoch  5  loss 0.3346
epoch 10  loss 0.3150
epoch 15  loss 0.3067


## Embedding + reconstruction error

In [3]:
model.eval()
embs, errs = [], []
with torch.no_grad():
    for i in range(0, len(Z), 8192):
        xb = torch.tensor(Z[i:i + 8192], device=dev)
        rec, z = model(xb)
        embs.append(z.cpu().numpy())
        errs.append(((rec - xb) ** 2).mean(1).cpu().numpy())
emb = np.concatenate(embs)
err = np.concatenate(errs)
out = pd.DataFrame(index=idx)
for i in range(emb.shape[1]):
    out[f"x_ae_{i}"] = emb[:, i]
out["x_ae_recon_error"] = err
out.shape

(356255, 17)

# Save

In [4]:
save_features(out, "autoencoder"); out.shape

(356255, 17)